# Aprobación

Usa aprobación nativa en Databricks y autoaprobación configurable en local.

In [ ]:
import os
from mlflow.tracking import MlflowClient
from iris_mlflow_utils import (
    approve_locally,
    build_deployment_config,
    build_runtime_config,
    detect_runtime,
)

runtime_mode = detect_runtime()
if runtime_mode == 'databricks':
    dbutils.widgets.text('promotion_profile', 'prod')
    os.environ['IRIS_PROMOTION_PROFILE'] = dbutils.widgets.get('promotion_profile')
config = build_runtime_config(model_slug='random_forest')
deployment_config = build_deployment_config()
if runtime_mode == 'databricks':
    dbutils.widgets.text('model_name', deployment_config.model_name)
    dbutils.widgets.text('model_version', '')
    model_name = dbutils.widgets.get('model_name')
    model_version = dbutils.widgets.get('model_version')
else:
    model_name = deployment_config.model_name
    model_version = os.getenv('IRIS_MODEL_VERSION', '')
if config.tracking_uri:
    import mlflow
    mlflow.set_tracking_uri(config.tracking_uri)
client = MlflowClient(registry_uri=config.registry_uri)
if not model_version:
    versions = list(client.search_model_versions(f"name='{model_name}'"))
    if not versions:
        raise RuntimeError(f'No hay versiones para {model_name}.')
    model_version = str(max(versions, key=lambda item: int(item.version)).version)
version = client.get_model_version(model_name, model_version)
if version.tags.get('evaluation_status') != 'passed':
    raise RuntimeError('La versión no tiene una evaluación aprobada.')
if runtime_mode == 'local':
    auto_approve = os.getenv('IRIS_LOCAL_AUTO_APPROVE', str(config.auto_approve)).lower() in {'1', 'true', 'yes'}
    if not auto_approve:
        raise RuntimeError('Configura IRIS_LOCAL_AUTO_APPROVE=true para simular aprobación.')
    approval = approve_locally(
        client, model_name=model_name, model_version=model_version,
        approval_tag=deployment_config.required_approval_tag,
    )
    print({'runtime': runtime_mode, 'model_name': model_name, 'model_version': model_version, **approval})
else:
    approval = version.tags.get(deployment_config.required_approval_tag, '')
    if approval != 'Approved':
        raise RuntimeError(f'Pendiente de aprobación: {deployment_config.required_approval_tag}=Approved')
    client.set_model_version_tag(model_name, model_version, 'approval_status', 'approved')
    dbutils.jobs.taskValues.set(key='approval_status', value='approved')
    print({'runtime': runtime_mode, 'model_name': model_name, 'model_version': model_version, 'approval': approval})
